# **Heart Rate Variability (HRV) Analysis**

Pipeline for conducting Heart Rate Variability (HRV) analysis on preprocessed ECG or PPG data, developed as part of the Brain-Body Analysis Special Interest Group (BBSIG).

To know how to use this HRV Analysis notebook, visit our step-by-step tutorial: https://martager.github.io/bbsig/hrv-analysis/

Jupyter notebook created by Mia Neubauer, Aleksandra Piejka, Niket Aggarwal, Marta Gerosa & Antonin Fourcade

Created on: 21 March 2025

Last update (by M. Gerosa): 11 April 2025

If you use this BBSIG pipeline in a publication, please cite us: *Gerosa M., Agrawal N., Ciston A.B., Fischer A., Fourcade A., Koushik A., Neubauer M., Patyczek A., Piejka A., Reinwarth E., Roellecke L., Shum Y.H., Verschooren S., Gaebler M. (2025). Brain-Body Analysis Special Interest Group (BBSIG) (Version 0.0.1) [Computer software]. https://martager.github.io/bbsig/*

## **Pipeline structure**
The following steps are included:

1. **Data import and conversion**: import and format RR intervals time series from either BBSIG preprocessed JSON files (`_ecg-preproc.json` or `ppg-preproc.json`) or custom TSV/CSV/TXT files (`_rr.{tsv/csv/txt}`), converting them to milliseconds (if needed) and computing R-peak timestamps stored in `rrs_dict` for later processing stages.
2. **(Optional) Data cropping**: crop RR interval data to a specific time window using the custom function `hrv_window_crop()`, enabling later analysis of a selected window (e.g., a 5-min block) based on a start timepoint (in seconds) plus either end timepoint (in seconds) or duration (in seconds). The cropped RR intervals and R-peak timestamps are then stored in a new `cropped_rrs_dict`.
3. **(Optional) Compute time-domain HRV metrics**: compute time-domain HRV metrics using the custom function `hrv_time_domain()` (based on NeuroKit2 `hrv_time()`), extracting values like mean/median RR and HR, SDNN, RMSSD, and pNN50, and storing them in a dictionary. Optionally, display a summary plot of time-domain HRV metrics if `show_plots=True`.
4. **(Optional) Compute frequency-domain HRV metrics**: compute frequency-domain HRV metrics using the custom function `hrv_frequency_domain()` (based on NeuroKit2 `hrv_frequency()`), extracting spectral features like VLF, LF, HF, LF/HF, LFn, HFn, and LnHF, and storing them in a dictionary. Optionally, display a summary plot of frequency-domain HRV metrics if `show_plots=True`.
5. **(Optional) Compute non-linear HRV metrics**: compute non-linear HRV metrics using the custom function `hrv_nonlinear()` (based on NeuroKit2 `hrv_nonlinear()`), extracting features like Poincaré plot SD1, SD2, SD1/SD2, ApEn and SampEn, and storing them in a dictionary. Optionally, display a summary plot of non-linear HRV metrics if `show_plots=True`.
6. **Data output**: save the computed HRV metrics (time-domain, frequency-domain, or non-linear; depending on which optional steps were enabled) of all participants to a summary TSV file in `derivatives/hrv-analysis/`, using the custom function `save_hrv_data()`. The summary TSV files for each HRV metrics can be distinguished using the suffix `_hrv_{time/freq/nonlinear}.tsv`.

All the preceding sections, depending on whether they are enabled in the optional pipeline steps, will be executed at once in the section **Main analysis loop** on all participant included in the `participant_ids` list. This will save the corresponding TSV output files with the computed HRV metrics, one row per participant. Please make sure that all the desired pipeline steps are set to `True` in the settings below.

In [ ]:
############## Import modules ##############

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import json
import os
import neurokit2 as nk

## **Settings: optional pipeline steps**

This section defines a series of variables that can be set to `True` if the corresponding pipeline step needs to be included:

| Variable name | Function |
| --- | --- |
| **`compute_hrv_time`** (bool) | **Time-domain HRV computation** (Sect. 3): computing time-domain HRV metrics, including the Standard Deviation of NN intervals (`SDNN`), Root Mean Square of Successive Differences (`RMSSD`) and Proportion of NN intervals > 50ms difference (`pNN50`), using NeuroKit2 `hrv_time()` function. |
| **`compute_hrv_freq`** (bool) | **Frequency-domain HRV computation** (Sect. 4): computing frequency-domain HRV metrics, such as Low Frequency (`LF`) and High Frequency (`HF`) power, Low-to-High Frequency Ratio (`LF/HF`), using NeuroKit2 `hrv_frequency()` function with default method 'Welch' and interpolation rate set to `interpolation_freq = 4`. |
| **`compute_hrv_nl`** (bool) | **Non-linear HRV computation** (Sect. 5): computing non-linear HRV metrics, such as Poincaré plot SD1, SD2, SD1/SD2, ApEn and SampEn, using NeuroKit2 `hrv_nonlinear()` function. |
| **`show_plots`** (bool) | **NeuroKit2 HRV plotting**: for each of the above HRV metrics, enabling NeuroKit2 in-built plotting. | 

Note: at least one of the two options `compute_hrv_time` or `compute_hrv_freq` needs to be true!

In [ ]:
############## Settings: optional pipeline steps ##############

# For S3: set whether (optional) time-domain HRV is needed
compute_hrv_time = True

# For S4: set whether (optional) frequency-domain HRV is needed
compute_hrv_freq = True

# For S5: set whether (optional) non-linear HRV is needed
compute_hrv_nl = True

# Plotting settings
show_plots = True

## **Additional settings for each pipeline step**

This section defines a series of settings specific to some pipeline steps, including:

- **Settings: 1. Data import and conversion**: 
    - **Define BIDS entities for data path info**: first, request the user to specify the participant ID(s) in the `participant_ids` list. The user has to specify mandatory (i.e., task `<label>` in format `task-<label>`, datatype) and optional (i.e., session `<label>` in format `ses-<label>`) BIDS entities, which will be used to create a base filename according to BIDS conventions (e.g., `sub-<ID>{_ses-<label>}_task-<label>`) and a base BIDS directory including subject, session (optional) and datatype (e.g., `'sub-<ID>/{ses-<label>}/<datatype>/'`). 
    - **Specify whether to use BBSIG preprocessed data or custom file**: 
        - **BBSIG preprocessed data**: if raw ECG/PPG data was preprocessed using the BBSIG pipelines, `bbsig_preproc` must be set to `True` (see below). During data import, the BSSIG-related preprocessed JSON files from the corresponding `derivatives\{physio_type}-preproc\` folder will be loaded. The user has to specify the type of physiological data on which preprocessing was performed (i.e., `physio_type = 'ecg'` or `physio_type = 'ppg'`) and the type of RR interval data to extract (i.e., `rr_type` set to either `'manualcorr'`, `'autocorr'`, or `'uncorr'`). 
        - **Custom RR intervals file**: if the BBSIG pipelines were not used for preprocessing, `bbsig_preproc` must be set to `False` (see below). During data import, a custom file containing RR intervals (either in seconds or milliseconds) for each participant will be loaded. The custom file should either be a TSV, CSV or TXT file, and include at least on column with the RR intervals time series, either with or without a header. The custom files for each participant should be named in a BIDS-compliant way, with the suffix `_rr.{tsv/csv/txt}`, and should be stored within the  `derivatives\sub-<label>\[ses-<label>]\<datatype>\` directory. The user has to specify the time units of the RR intervals (i.e., `rri_unit = 's'` or `rri_unit = 'ms'`), the type of custom file to load (i.e., `file_type` set to either `'tsv'`, `'csv'` or `'txt'`), the header row index (if present, `file_header`; 0-based) and the column index containing the RR intervals (i.e., `column_rr`; 0-based). 

- **Settings: 2. (Optional) Data cropping**: if `crop_window` is set to `True` (see below), the user has to specify parameters for cropping a window from the original RR intervals time series. It is mandatory to specify the start time of the cropping window (in seconds; `window_start_hrv`), while it could be chosen whether to specify the end time of the cropping window (in seconds; `window_end_hrv`), or alternatively the cumulative duration of the cropping window (in seconds; `window_length_hrv`). One out of `window_end_hrv` and `window_length_hrv` has to be specified, while the other has to be set to `None`. If enabled, metadata regarding the chosen cropped window start and end timepoint will be saved in a new `cropped_rrs_dict` for each participant. 

- **Settings: 4. Frequency-domain HRV settings**: if `compute_hrv_freq` is set to `True` (see above), the user can specify the sampling frequency for interpolating RR intervals for frequency-domain HRV analysis. Default is `interpolation_freq = 4`. 

In [ ]:
############## Other settings for each pipeline step ##############

###### Settings: 1. Data import and conversion ######

### Define BIDS entities for data path info ###

# Define the participant IDs
participant_ids = ['101', '103']  # Adjust as needed: it should correspond to a list of <ID> of 'sub-<ID>' in BIDS format

# Specify the main directory of data storage (containing BIDS-compatible raw and derivatives folder)
wd = r'C:\Users\gerosa\Desktop\BBSIG_datasets\ECG' # change with the directory of data storage

# Mandatory: BIDS entities (task, datatype)
task_name = 'BBSIG'       # <label> of 'task-<label>' used for file naming in BIDS format
datatype_name = 'beh'     # datatype used for corresponding directory in BIDS format (e.g., 'beh', 'eeg', 'func')

# Optional: BIDS entities (session)
session_idx = '1'     # <label> of 'ses-<label>' in BIDS format, if available; otherwise, set to None


### Specify whether the data is BBSIG preprocessed output or custom file ###

# Define whether the preprocessed ECG/PPG data were generated using the BBSIG pipeline
bbsig_preproc = True  # set to True if data from the 'ecg-preproc' or 'ppg-preproc' folder is used
rri_unit = 's'      # RR interval unit; 's' for seconds, 'ms' for milliseconds

# Parameters for `bbsig_preproc = True`: BBSIG-compatible preprocessed file
if bbsig_preproc: 
    physio_type = 'ecg'  # specify the type of physiological data ('ecg', 'ppg') if bbsig_preproc was used
    rr_type = 'autocorr' # specify the type of RR interval data to extract (either 'manualcorr', 'autocorr', or 'uncorr')

# Parameters for `bbsig_preproc = False`: custom file 
if not bbsig_preproc: 
    file_type = 'csv'   # specify the file type ('tsv', 'csv', 'txt') for loading data
    file_header = None  # set to None if no header, otherwise 0 if header is first row (0-based)
    column_rr = 0       # specify the column index (0-based) for RR intervals

In [ ]:
###### Settings: 2. (Optional) Data cropping ######

# Define whether cropping of a window of specified duration or start/end time is needed
crop_window = True  # set to True if needed

# Specify the parameters below if crop_window is set to True
# one out of window_end_hrv and window_length_hrv has to be specified, the other can be set to None
if crop_window: 
    window_start_hrv = 0  # mandatory: start of window for HRV analysis (in seconds)
    window_end_hrv = 90  # optional: end of window for HRV analysis (in seconds), otherwise set to None
    window_length_hrv = None  # window length for HRV analysis (in seconds), otherwise set to None

In [ ]:
####### Settings: 4. Frequency-domain HRV settings #######

# Specify the parameters below if compute_hrv_freq is set to True
if compute_hrv_freq: 
    interpolation_freq = 4  # sampling frequency for interpolating RR intervals for frequency-domain HRV analysis

## **1. Data import and conversion**

This section defines a custom function `load_rr_data()` to import the RR intervals time series from either the ECG/PPG data preprocessed using the BBSIG pipelines (`bbsig_preproc = True`) or a custom TSV/CSV/TXT file (`bbsig_preproc = False`) for a given participant, and convert them in the appropriate format for later processing stages (RR intervals in ms, R-peaks timestamps). 

The function supports two data input modes:

* **From preprocessed JSON files generated by the BBSIG pipelines** (`_ecg-preproc.json` or `_ppg-preproc.json`, depending on the peripheral physiological data type) in the `derivatives` folder.
    - Depending on what has been specified, extracts the manual, automatic, or uncorrected R-peak types (`rr_type`) from the JSON file.
* **From custom TSV/CSV/TXT files** containing at least one column with the RR intervals time series (`_rr.{tsv/csv/txt}`) in the `derivatives` folder. 
    - Depending on what has been specified, extracts the RR intervals time series from a given column (`column_rr`), excluding the header row, if present (`file_header`).

After loading the corresponding RR intervals from either the BBSIG-generated JSON file or the custom file, the function:

- Converts RR interval values from seconds to milliseconds (if needed)
- Computes of R-peak timestamps as the cumulative sum of RR intervals 
- Stores a dictionary `rrs_dict` containing RR intervals in ms (`RRI`) and R-peaks timestamps (`RRI_Time`), in a format compatible with later NeuroKit2 functions. 

In [ ]:
############## 1. Data import and conversion ##############

# Define function to import RR intervals time series from either BBSIG-generated preprocessed data file or custom file
def load_rr_data(wd, bids_base_fname, bids_base_dir, rr_type, rri_unit, bbsig_preproc=True, physio_type='ecg'):
    """ Load RR interval data from BBSIG-generated preprocessed data file or custom file.

    Parameters: 
    - wd (str): Root directory containing the BIDS-formatted dataset.
    - bids_base_fname (str): BIDS-compatible base filename (e.g., 'sub-01[_ses-01]_task-rest').
    - bids_base_dir (str): Relative BIDS-compatible base directory for the subject/session/datatype (e.g., 'sub-01/[ses-01]/beh').
    - rr_type (str): If 'bbsig_preproc=True', type of R-peak correction to use: 'manualcorr', 'autocorr', or 'uncorr'.
    - rri_unit (str): Unit of RR intervals, either 's' (seconds) or 'ms' (milliseconds).
    - bbsig_preproc (bool, optional): If True, load data from a BBSIG-generated JSON file. If False, load from a custom RR interval file.
    - physio_type (str, optional): If 'bbsig_preproc=True', type of physiological signal used. Default is 'ecg', can also be 'ppg'.

    Returns: 
    - A dictionary with two keys: 'RRI' (array of RR intervals in ms); 'RRI_Time' (array of cumulative R-peaks timestamps in seconds). 
    """

    ############## Data loading: BBSIG preprocessing files ##############

    # Data import got 
    if bbsig_preproc:

        # Specify the name and directory of the BIDS-compatible ecg preprocessed data file
        bbsig_preproc_json_fname = f'{bids_base_fname}_{physio_type}-preproc.json'
        bbsig_preproc_json_fpath = os.path.join(wd, 'derivatives', f'{physio_type}-preproc', 
                                                bids_base_dir, bbsig_preproc_json_fname)
    
        # Extract RR intervals from ecg-preproc JSON file
        with open(bbsig_preproc_json_fpath, 'r') as f:
            data = json.load(f)

        # Check the type of R-peak correction ('manualcorr', 'autocorr', or 'uncorr')
        if rr_type == 'manualcorr':
            rr_name = 'RR_s_ManualCorr'
        elif rr_type == 'autocorr':
            rr_name = 'RR_s_AutoCorr'
        elif rr_type == 'uncorr':
            rr_name = 'RR_s_Uncorr'
        else:
            raise ValueError('Invalid RR interval type. Choose from "manualcorr", "autocorr", or "uncorr".')
        
        # Extract RR intervals according to specified R-peak correction type
        rr_s = data.get('rr_s', {}).get(rr_name, None)
        rr_s = np.array(rr_s).flatten() 

        print(f"Loading {physio_type.upper()} preprocessed file: {bbsig_preproc_json_fname}")

    ############## Data loading: other customized files ##############

    else:
        # If bbsig_preproc not used, load a customized file with suffix '_rr.tsv', '_rr.csv' or '_rr.txt'
        rr_fname = f'{bids_base_fname}_rr.{file_type}'
        rr_fpath = os.path.join(wd, 'derivatives', bids_base_dir, rr_fname)
        
        # Load the RR intervals file (`_rr.{tsv/csv/txt}`) according to its extension
        if file_type == 'csv':
            sep_type = ','
        elif file_type == 'tsv':
            sep_type = '\t'
        elif file_type == 'txt':
            sep_type = ' '
        else:
            raise ValueError('Unsupported file type. Choose from "csv", "tsv", or "txt".')
        
        # Extract RR intervals according to specified separator type, header and column index
        rr_s = pd.read_csv(rr_fpath, sep=sep_type, header=file_header, usecols=[column_rr])
        rr_s = rr_s.values.flatten()

        print(f"Loading custom RR intervals file: {rr_fname}")

    # Convert RR intervals from seconds to milliseconds, if necessary
    if rri_unit == 's':
        rr_ms = rr_s * 1000
    else:
        rr_ms = rr_s

    # Convert data into appropriate formats for NeuroKit2
    # NeuroKit2 expects RRI in milliseconds and RRI_Time in seconds.
    peaks_s = np.cumsum(rr_ms) / 1000  # R-peak timestamps in seconds

    # Store data for participant in a dictionary
    rrs_dict = {
        'RRI': np.array(rr_ms),  # RR intervals in milliseconds
        'RRI_Time': peaks_s}     # R-peaks timestamps in seconds
               
    return rrs_dict

## **2. (Optional) Data cropping**

If `crop_window` is set to `True`, this section defines a custom function `hrv_window_crop()` to crop RR interval data to a specific time window, which is useful to analyze a subset of a recording (e.g., a 5-min rest block) instead of the entire session. This function allows to extract a specific segment of RR interval data based on either a start and end time (in seconds), or a start time and a duration (in seconds). The corresponding RR intervals and R-peaks timestamps for the cropped window are stored in a  new `cropped_rrs_dict` dictionary. 

Note: either one of `window_end` and `window_length` has to specified, while the other has to be set to `None`. 

In [ ]:
############## 2. (Optional) Data cropping ##############

# Define function to crop a window of specified duration (start time + end time, or start time + length)
def hrv_window_crop(rrs_dict, window_start=0, window_end=None, window_length=None):
    """ Crop RR interval time series to a specified time window for HRV analysis.
    Parameters:
    - rrs_dict (dict): Dictionary storing RR intervals in ms and R-peaks timestamps in seconds, generated by load_rr_data().
    - window_start (float): Start time of the window to be cropped (in seconds).
    - window_end (float, optional): End time of the window to be cropped (in seconds).
    - window_length (float, optional): Length of the window to be cropped (in seconds).

    Returns:
    - cropped_rrs_dict (dict): Cropped RR interval data.
    - window_start (float): Actual start time of the window.
    - window_end (float): Actual end time of the window.

    Raises:
    - ValueError: If both or neither of `window_end` and `window_length` are provided.
    """

    # Check if 3 arguments are provided
    if window_end is not None and window_length is not None:
        raise ValueError('Please provide either `window_end` or `window_length`, not both.')
    # Check if at least 2 arguments are provided
    if window_end is None and window_length is None:
        raise ValueError('Please provide either `window_end` or `window_length`.')
    
    # If window_length is provided, begin from window_start to calculate the window_end according specified length
    if window_end is None:
        window_end = window_start + window_length

    # If window_end is provided, calculate window_length by subtracting window_start from window_end
    if window_length is None:
        window_length = window_end - window_start

    # Selected the R-peaks comprised between window_start and window_end
    rrs_dict['RRI_Time'] = rrs_dict['RRI_Time'].tolist()
    rrt_cropped = [x for x in rrs_dict['RRI_Time'] if x >= window_start and x <= window_end]

    # Get indices of window_start and window_end and extract RR interval durations comprised in-between
    window_start_idx = rrs_dict['RRI_Time'].index(rrt_cropped[0])
    window_end_idx = rrs_dict['RRI_Time'].index(rrt_cropped[-1])
    rri_cropped = rrs_dict['RRI'][window_start_idx:window_end_idx+1]

    # Store data from cropped window for given participant
    cropped_rrs_dict = {
        'RRI': np.array(rri_cropped),           # RR intervals in ms
        'RRI_Time': np.array(rrt_cropped)}      # R-peaks timestamps in s
    
    return cropped_rrs_dict, window_start, window_end

## **3. (Optional) Compute time-domain HRV metrics**

If the variable `compute_hrv_time` is set to `True` in the optional pipeline steps (see settings above), this section defines a custom function `hrv_time_domain()` that:

- Performs the calculation of time-domain HRV metrics, using NeuroKit2 `hrv_time()` function.
- Extracts and formats the resulting time-domain metrics for the given participant in `hrv_time_metrics` dictionary, including:
    - `MeanRR` and `MeanBPM`: mean RR duration in seconds and mean heart rate (HR) value in beats per minute (bpm)
    - `MedianRR` and `MedianBPM`: median RR duration in seconds and median HR value in bpm
    - `MinRR` and `MinBPM`: minimum RR duration in seconds and minimum HR in bpm
    - `MaxRR` and `MaxBPM`: maximum RR duration in seconds and maximum HR in bpm
    - `SDNN`: Standard Deviation of NN (normal-to-normal) intervals
    - `SDSD`: Standard Deviation of Successive RR interval Differences 
    - `RMSSD`: Root Mean Square of Successive Differences 
    - `pNN50`: Proportion of successive NN intervals that differ more than 50ms
- If `show_plots` is set to `True` in the optional pipeline steps (see settings above), a plot for the computed time-domain HRV metrics will also be shown. 

In [ ]:
############## 3. (Optional) Compute time-domain HRV metrics ##############

# Define function to compute and store time-domain HRV metrics
def hrv_time_domain(rrs_dict, subj, plot=False):
    
    # Calculate Time-domain HRV metrics, using NeuroKit2
    hrv_t = nk.hrv_time(rrs_dict, show=show_plots)

    # Extract and format metrics
    hrv_time_metrics = {
        'subjID': f'sub-{subj}',
        'MeanRR': hrv_t['HRV_MeanNN'][0],
        'MeanBPM': 60000 / hrv_t['HRV_MeanNN'][0],
        'MedianRR': hrv_t['HRV_MedianNN'][0],
        'MedianBPM': 60000 / hrv_t['HRV_MedianNN'][0],
        'MinRR': hrv_t['HRV_MinNN'][0],
        'MinBPM': 60000 / hrv_t['HRV_MaxNN'][0],
        'MaxRR': hrv_t['HRV_MaxNN'][0],
        'MaxBPM': 60000 / hrv_t['HRV_MinNN'][0],
        'SDNN': hrv_t['HRV_SDNN'][0],
        'SDSD': hrv_t['HRV_SDSD'][0],
        'RMSSD': hrv_t['HRV_RMSSD'][0],
        'pnn50': hrv_t['HRV_pNN50'][0]      
    }
    return hrv_time_metrics

## **4. (Optional) Compute frequency-domain HRV metrics**

If the variable `compute_hrv_freq` is set to `True` in the optional pipeline steps (see settings above), this section defines a custom function `hrv_frequency_domain()` that: 

- Performs the calculation of frequency-domain HRV metrics, using NeuroKit2 `hrv_frequency()` function with default method 'Welch' and interpolation at 4Hz (yet, the interpolation rate can be changed by specifying the `interpolation_rate` variable in the settings above). 
- Extracts and formats the resulting frequency-domain metrics for the given participant in `hrv_freq_metrics` dictionary, including:
    - `VLF`: Very Low Frequency power (by default, 0.0033 to 0.04 Hz)
    - `LF`: Low Frequency power (by default, 0.04 to 0.15 Hz)
    - `HF`: High Frequency power (by default, 0.15 to 0.4 Hz)
    - `LF/HF`: ratio of Low-to-High Frequency power — often interpreted as a marker of autonomic balance
    - `LFn`: normalized Low Frequency power, obtained by dividing the low frequency power by the total power
    - `HFn`: normalized High Frequency power, obtained by dividing the low frequency power by the total power
    - `LnHF`: natural logarithm of HF power
- If `show_plots` is set to `True` in the optional pipeline steps (see settings above), a plot for the computed frequency-domain HRV metrics will also be shown. 

In [ ]:
############## 4. (Optional) Compute frequency-domain HRV metrics ##############

# Define function to compute and store frequency-domain HRV metrics
def hrv_frequency_domain(rrs_dict, subj, interpolation_freq=4, plot=False):
    # Calculate Frequency-domain HRV metrics using NeuroKit2
    # Using Welch method and interpolation at 4Hz by default
    # If plot is needed, set show=True
    # ulf=(0, 0) disables the ULF band, which otherwise returns spurious values
    # for short (<5 min) recordings and dominates the PSD plot.
    hrv_f = nk.hrv_frequency(rrs_dict, interpolation_rate=interpolation_freq,
                             ulf=(0, 0),
                             normalize=False, psd_method='welch', show=show_plots) 
    
    # Extract and format metrics
    hrv_freq_metrics = {
        'subjID': f'sub-{subj}',
        'VLF': hrv_f['HRV_VLF'][0],
        'LF': hrv_f['HRV_LF'][0],
        'HF': hrv_f['HRV_HF'][0],
        'LF/HF': hrv_f['HRV_LFHF'][0],
        'LFn': hrv_f['HRV_LFn'][0],
        'HFn': hrv_f['HRV_HFn'][0],
        'LnHF': hrv_f['HRV_LnHF'][0]

    } 
    return hrv_freq_metrics, hrv_f

## **5. (Optional) Compute non-linear HRV metrics**

If the variable `compute_hrv_nl` is set to `True` in the optional pipeline steps (see settings above), this section defines a custom function `hrv_nonlinear()` that: 

- Performs the calculation of non-linear HRV metrics, using NeuroKit2 `hrv_nonlinear()` function.
- Extracts and formats the resulting non-linear metrics in `hrv_ln_metrics` dictionary, including:
    - `SD1`: Poincaré plot standard deviation of the short-term variability component (short-axis of ellipse).
    - `SD2`: Poincaré plot standard deviation of the long-term variability component (long-axis of ellipse).
    - `SD1/SD2`: Poincaré ratio of short-term to long-term variability. 
    - `S`: Poincaré plot area of the ellipse which represents total HRV
    - `ApEn`: approximate entropy, a measure of regularity and complexity of a time series.
    - `SampEn`: sample entropy, a more robust version of ApEn.
- If `show_plots` is set to `True` in the optional pipeline steps (see settings above), a plot for the computed non-linear HRV metrics will also be shown. 

In [ ]:
############## 5. (Optional) Compute non-linear HRV metrics ##############

# Define function to compute and store non-linear HRV metrics
def hrv_nonlinear(rrs_dict, subj, plot=False):

    # Calculate Nonlinear HRV metrics using NeuroKit2
    hrv_nl = nk.hrv_nonlinear(rrs_dict, show=show_plots) 
    
    # Extract and format metrics
    hrv_nl_metrics = {
        'subjID': f'sub-{subj}',
        'SD1': hrv_nl['HRV_SD1'][0],
        'SD2': hrv_nl['HRV_SD2'][0],
        'SD1/SD2': hrv_nl['HRV_SD1SD2'][0],
        'S': hrv_nl['HRV_S'][0],
        'ApEn': hrv_nl['HRV_ApEn'][0],
        'SampEn': hrv_nl['HRV_SampEn'][0]
    }
    return hrv_nl_metrics

## **6. Data Output**

This section defines a custom function `save_hrv_data()` that saves the computed HRV metrics (time-domain, frequency-domain, or non-linear; depending on which optional steps were enabled) of all participants to a TSV file. In detail: 

- Converts the input `hrv_data` dictionary (generated from one of the pipelines steps in Sect. 3, Sect. 4 or Sect. 5) into a pd DataFrame
- Creates an output directory at `derivatives/hrv-analysis/` if it does not already exist.
- Saves the DataFrame as a TSV file named according to the task and HRV analysis type (`time`, `freq` or `nonlinear`), e.g., `task-rest_hrv_time.tsv`. 

In [ ]:
def save_hrv_data(wd, task_name, session_idx, hrv_data, hrv_type):

    # Save HRV data to a TSV file in 'derivatives/hrv-analysis/' folder
    hrv_data_df = pd.DataFrame(hrv_data)
    output_dir = os.path.join(wd, 'derivatives', 'hrv-analysis')
    if not os.path.exists(output_dir):
        os.makedirs(output_dir)

    if session_idx: 
        hrv_data_df.to_csv(os.path.join(output_dir, f'ses-{session_idx}_task-{task_name}_hrv_{hrv_type}.tsv'), 
                                        index=False, sep='\t')
    else:
        hrv_data_df.to_csv(os.path.join(output_dir, f'task-{task_name}_hrv_{hrv_type}.tsv'), 
                                        index=False, sep='\t')

## **Main analysis loop**

All the preceding sections, depending on whether they have been enabled in the optional pipeline steps, are executed at once in this main analysis loop on all participant included in the `participant_ids` list. 

This will save the corresponding TSV output files with the computed HRV metrics, one row per participant. Please make sure that all the desired pipeline steps are set to `True` in the settings above. Note that at least one of the two options `compute_hrv_time` or `compute_hrv_freq` needs to be `True`!

In [ ]:
# initialize empty lists to store HRV metrics
all_hrv_time = []
all_hrv_freq = []
all_hrv_nl = []

# Loop through each participant ID and perform HRV analysis
for subj in participant_ids:

    subj_id = 'sub-' + str(subj) # participant ID (in BIDS format)
    
    ############## Create base BIDS-compatible filename and directory ##############

    # If you have additional BIDS entities (e.g., 'run' or 'recording') you can change the base BIDS file name below
    # e.g., f'{subj_id}_ses-{session_idx}_task-{task_name}_run-{run_idx}_recording-{rec_name}' 
    if session_idx is not None:
        bids_base_fname = f'{subj_id}_ses-{session_idx}_task-{task_name}' 
        bids_base_dir = os.path.join(subj_id, f'ses-{session_idx}', datatype_name) # base BIDS folders incl. session 'sub-<ID>/ses-<label>/<datatype>/'
    else:
        bids_base_fname = f'{subj_id}_task-{task_name}'
        bids_base_dir = os.path.join(subj_id, datatype_name) # base BIDS folders without session 'sub-<ID>/<datatype>/' 


    ############## Sect. 1: Data import of RR intervals ##############
    rr = load_rr_data(wd, bids_base_fname, bids_base_dir, rr_type, rri_unit, bbsig_preproc=bbsig_preproc, physio_type=physio_type)


    ############## Sect. 2: Data cropping ##############
    if crop_window:
        rr, window_start, window_end = hrv_window_crop(rr, window_start=window_start_hrv, window_end=window_end_hrv, window_length=window_length_hrv)
    

    ############## Sect. 3: Computation of time-domain HRV, if required ##############
    if compute_hrv_time:
        hrv_time_metrics = hrv_time_domain(rr, subj, plot=show_plots)
        if crop_window:
            hrv_time_metrics['window_start'] = window_start     # save metadata about start of cropped window
            hrv_time_metrics['window_end'] = window_end         # save metadata about end of cropped window
        all_hrv_time.append(hrv_time_metrics)


    ############## Sect. 4: Computation of frequency-domain HRV, if required ##############
    if compute_hrv_freq:
        hrv_freq_metrics, hrv_f = hrv_frequency_domain(rr, subj, interpolation_freq, plot=show_plots)
        if crop_window:
            hrv_freq_metrics['window_start'] = window_start     # save metadata about start of cropped window
            hrv_freq_metrics['window_end'] = window_end         # save metadata about end of cropped window
        all_hrv_freq.append(hrv_freq_metrics)

    
    ############## Sect. 5: Computation of non-linear HRV, if required ##############
    if compute_hrv_nl:
        hrv_nl_metrics = hrv_nonlinear(rr, subj, plot=show_plots)
        if crop_window:
            hrv_nl_metrics['window_start'] = window_start       # save metadata about start of cropped window
            hrv_nl_metrics['window_end'] = window_end           # save metadata about end of cropped window
        all_hrv_nl.append(hrv_nl_metrics)


############## Sect. 6: Data output ##############

# If enabled, save TSV file with computed time-domain HRV metrics for all participants
if compute_hrv_time:
    save_hrv_data(wd, task_name, session_idx, all_hrv_time, 'time')
    print(pd.DataFrame(all_hrv_time))

# If enabled, save TSV file with computed frequency-domain HRV metrics for all participants
if compute_hrv_freq:
    save_hrv_data(wd, task_name, session_idx, all_hrv_freq, 'freq')
    print(pd.DataFrame(all_hrv_freq))

# If enabled, save TSV file with computed non-linear HRV metrics for all participants
if compute_hrv_nl:
    save_hrv_data(wd, task_name, session_idx, all_hrv_nl, 'nonlinear')
    print(pd.DataFrame(all_hrv_nl))

## **Good job, your HRV analysis is done!**

When using or adapting this BBSIG pipeline to conduct HRV analysis in your research work, please cite us in your publication as follows: 

**APA**

*Gerosa M., Agrawal N., Ciston A.B., Fischer A., Fourcade A., Koushik A., Neubauer M., Patyczek A., Piejka A., Reinwarth E., Roellecke L., Shum Y.H., Verschooren S., Gaebler M. (2025). Brain-Body Analysis Special Interest Group (BBSIG) (Version 0.0.1) [Computer software]. https://doi.org/10.5281/zenodo.15212797*